# TP3 — XGBoost : prédire les hauts revenus (Énoncé)

Cours associé : [TP3_Cours.md](TP3_Cours.md) · Pense-bête : [Pense_bete_Pandas.md](../Pense_bete_Pandas.md)

## Contexte

Le jeu de données *Adult* est extrait du recensement américain de 1994. Pour 48 842 adultes, il décrit l'âge, le niveau d'études, la profession, la situation familiale, le temps de travail… et indique si leur revenu annuel dépasse **50 000 $**.

**Objectif** : prédire la classe `>50K`, en tirant le meilleur d'XGBoost, puis comprendre ce que le modèle a appris.

Comme au TP2, on y trouve des variables catégorielles nombreuses, des valeurs manquantes et des classes déséquilibrées (24 % de hauts revenus), mais avec sept fois plus de lignes.

## Déroulé

| Partie | Contenu |
|---|---|
| 1 | Chargement et préparation |
| 2 | Modèles de référence : régression logistique, arbre, forêt aléatoire |
| 3 | Premier modèle XGBoost |
| 4 | Arrêt anticipé |
| 5 | Interprétation, évaluation finale et synthèse |
| 6 | *Facultatif* : classes déséquilibrées |
| 7 | *Facultatif* : taux d'apprentissage et hyperparamètres |

Les cellules marquées `# TODO` sont à compléter. Les cellules ✍️ attendent une réponse rédigée.

⚙️ XGBoost fait partie du `requirements.txt` du cours. S'il manque dans votre environnement :

- avec **uv** : `uv pip install xgboost`
- avec **Conda** : `conda install -c conda-forge xgboost`

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report)

import xgboost as xgb
from xgboost import XGBClassifier

plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42
print("xgboost", xgb.__version__)

## Partie 1 — Chargement et préparation

**1.1** Chargez `data/adult.csv.gz` dans `df` (pandas décompresse automatiquement). Affichez les dimensions, les types, les valeurs manquantes et la répartition de la colonne `class`.

In [ ]:
# TODO : chargement et premier coup d'œil


**1.2** Préparez les données :

- créez la cible `y` valant 1 si `class == ">50K"`, 0 sinon ;
- supprimez de `X` les colonnes `class`, `fnlwgt` (un poids statistique du recensement, pas une caractéristique de la personne) et `education` (redondante avec `education-num`, qui code le même niveau d'études sous forme numérique) ;
- convertissez toutes les colonnes de type texte de `X` en type `category` (nécessaire pour qu'XGBoost les gère nativement).

Affichez la liste des colonnes catégorielles et numériques.

In [ ]:
# TODO : cible, suppression de colonnes, conversion en category


**1.3** Découpez les données en trois jeux stratifiés : **entraînement 60 %**, **validation 20 %**, **test 20 %** (deux appels successifs à `train_test_split`). Le jeu de validation servira à l'arrêt anticipé et aux réglages ; le jeu de test uniquement à l'évaluation finale.

In [ ]:
# TODO : découpage train / valid / test


## Partie 2 — Modèles de référence

Avant d'essayer XGBoost, il faut savoir d'où l'on part. Pour comparer les modèles, on utilisera l'**AUC** (insensible au seuil et adaptée au déséquilibre) et, en complément, l'exactitude et le F1.

Deux outils sont **fournis** ci-dessous :

- la fonction `evaluer`, qui calcule ces métriques sur le jeu de validation et les range dans un tableau ;
- le préprocesseur `preprocess`, identique à celui du TP2 : les modèles de scikit-learn ne gèrent ni les catégories ni les valeurs manquantes, il faut donc imputer, encoder et standardiser.

In [ ]:
resultats = {}

def evaluer(nom, modele, X_eval=None, y_eval=None, duree=None):
    """Calcule les métriques sur le jeu de validation et les mémorise dans `resultats`."""
    X_eval = X_valid if X_eval is None else X_eval
    y_eval = y_valid if y_eval is None else y_eval
    proba = modele.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    resultats[nom] = {
        "AUC": roc_auc_score(y_eval, proba),
        "exactitude": accuracy_score(y_eval, pred),
        "F1": f1_score(y_eval, pred),
        "durée (s)": duree,
    }
    return pd.DataFrame(resultats).T.round(3)

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat_cols),
])

**2.1** Évaluez, sur le jeu de **validation**, un `DummyClassifier`, une régression logistique et un arbre de décision (`max_depth=8`). Chaque modèle est placé dans un `Pipeline` derrière une copie du préprocesseur (`clone(preprocess)`). Mesurez le temps d'entraînement avec `time.perf_counter()`.

In [ ]:
# TODO : trois modèles de référence


**2.2** Ajoutez une forêt aléatoire (`RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1)`).

In [ ]:
# TODO : forêt aléatoire


✍️ **Votre réponse :**

…

## Partie 3 — Premier modèle XGBoost

**3.1** Entraînez un `XGBClassifier` avec ses réglages par défaut, en ajoutant seulement `enable_categorical=True`, `tree_method="hist"` et `random_state=RANDOM_STATE`. **Aucun préprocesseur n'est nécessaire** : XGBoost lit directement les colonnes `category` et les valeurs manquantes. Évaluez-le.

In [ ]:
# TODO : XGBoost par défaut


**3.2** Par défaut, XGBoost construit **100 arbres** de **profondeur 6** avec un **taux d'apprentissage de 0,3**. Comparez ce premier modèle aux modèles de référence : performance, temps d'entraînement, préparation des données nécessaire. Le taux d'apprentissage par défaut vous paraît-il prudent (voir la section 6.2 du cours) ?

✍️ **Votre réponse :**

…

## Partie 4 — Arrêt anticipé

**4.1** Entraînez un XGBoost avec `n_estimators=3000`, `learning_rate=0.05`, `max_depth=6`, `eval_metric="logloss"` en passant `eval_set=[(X_train, y_train), (X_valid, y_valid)]` à `fit` (avec `verbose=500`). **Sans** arrêt anticipé pour l'instant.

Récupérez les courbes de perte avec `evals_result()` et tracez la log loss d'entraînement et de validation en fonction du nombre d'arbres. Que constatez-vous ?

In [ ]:
# TODO : courbes d'apprentissage sans arrêt anticipé


✍️ **Votre réponse :**

…

**4.2** Reprenez le même modèle en ajoutant `early_stopping_rounds=50` dans le constructeur, et seulement le jeu de validation dans `eval_set`. Combien d'arbres sont retenus (`best_iteration`) ? Évaluez le modèle.

In [ ]:
# TODO : arrêt anticipé


✍️ **Votre réponse :**

…

Pour la suite du TP, on retient le modèle de la question 4.2 (taux d'apprentissage de 0,05 et arrêt anticipé).

In [ ]:
xgb_final = xgb_es

## Partie 5 — Interprétation, évaluation finale et synthèse

**5.1** Tracez l'importance des variables de `xgb_final` selon deux définitions : `gain` et `weight` (`xgb_final.get_booster().get_score(importance_type=...)`). Les classements sont-ils les mêmes ?

In [ ]:
# TODO : importance des variables


**5.2** Les trois premières lignes, fournies, calculent les valeurs SHAP du jeu de validation (voir la section 8.2 du cours) : `shap_df` contient une ligne par individu et une colonne par variable. Tracez l'importance moyenne `|SHAP|` par variable. Enfin, affichez la décomposition de la prédiction pour **un** individu de votre choix : quelles variables poussent vers `>50K`, lesquelles vers `<=50K` ?

In [ ]:
dvalid = xgb.DMatrix(X_valid, enable_categorical=True)
contribs = xgb_final.get_booster().predict(dvalid, pred_contribs=True, iteration_range=(0, xgb_final.best_iteration + 1))
shap_df = pd.DataFrame(contribs[:, :-1], columns=X_valid.columns, index=X_valid.index)
# TODO : importance moyenne |SHAP|, puis décomposition pour un individu

✍️ **Votre réponse :**

…

**5.3** Évaluation finale. Affichez le tableau `resultats` complet. Puis évaluez **une seule fois** le modèle retenu (`xgb_final`, seuil 0,5) sur le jeu de **test**.

In [ ]:
# TODO : tableau récapitulatif et évaluation finale sur le test


✍️ **Votre réponse :**

…

**5.4** Synthèse. Un collègue vous demande : « J'ai un fichier clients et je veux prédire qui va résilier. Est-ce que je prends XGBoost, et à quoi dois-je faire attention ? » Répondez en quelques phrases, en vous appuyant sur ce que vous avez observé dans ce TP.

✍️ **Votre réponse :**

…

## Partie 6 — *Facultatif* : classes déséquilibrées

*À faire si vous avez terminé le reste.*

**6.1** Affichez le `classification_report` de `xgb_final` sur le jeu de validation. Quelle classe est la moins bien reconnue ?

In [ ]:
# TODO : rapport de classification


**6.2** Entraînez une variante `xgb_pondere` identique à `xgb_final` mais avec `scale_pos_weight` égal au rapport nombre de négatifs / nombre de positifs dans `y_train`. Comparez précision, rappel, F1 et AUC des deux modèles.

In [ ]:
# TODO : pondération de la classe positive


✍️ **Votre réponse :**

…

## Partie 7 — *Facultatif* : taux d'apprentissage et hyperparamètres

*À faire si vous avez terminé le reste.* Jusqu'ici, on a fixé le taux d'apprentissage à 0,05 et laissé l'arrêt anticipé choisir le nombre d'arbres. Les autres réglages (sections 6.2 et 6.4 du cours) valent-ils la peine d'être ajustés ?

**7.1** Comparez trois taux d'apprentissage : 0,3, 0,1 et 0,02 (toujours avec arrêt anticipé). Pour chacun, affichez le nombre d'arbres retenus, l'AUC de validation et le temps d'entraînement. Concluez.

In [ ]:
# TODO : comparaison de taux d'apprentissage


✍️ **Votre réponse :**

…

**7.2** Lancez une recherche aléatoire (`RandomizedSearchCV`, 20 combinaisons, validation croisée à 3 plis, `scoring="roc_auc"`) sur le jeu d'**entraînement** avec l'espace suivant, pour un XGBoost à `n_estimators=300` et `learning_rate=0.1` :

- `max_depth` : 3 à 9 ;
- `min_child_weight` : 1, 3, 5, 10 ;
- `subsample` : 0,6 à 1 ;
- `colsample_bytree` : 0,5 à 1.

*Cela prend de quelques secondes à une minute selon votre machine. `n_jobs=-1` dans `RandomizedSearchCV` parallélise les calculs.*

In [ ]:
# TODO : recherche aléatoire


**7.3** Affichez les 5 meilleures combinaisons (`cv_results_`). L'écart entre elles est-il important ?

In [ ]:
# TODO : meilleures combinaisons


**7.4** Entraînez un modèle `xgb_regle` avec les meilleurs hyperparamètres trouvés, un `learning_rate` de 0,03, `n_estimators=5000` et l'arrêt anticipé sur le jeu de validation. Évaluez-le et comparez-le au modèle retenu en partie 4.

In [ ]:
# TODO : modèle réglé


✍️ **Votre réponse :**

…

## Pour aller plus loin (bonus)

- Sans pondération, cherchez le **seuil de décision** qui maximise le F1 sur le jeu de validation (seuils de 0,1 à 0,9 par pas de 0,01) et tracez précision, rappel et F1 en fonction du seuil.
- Mesurez l'**équité** du modèle : comparez le rappel et le taux de prédictions positives entre hommes et femmes.
- Remplacez XGBoost par `HistGradientBoostingClassifier` de scikit-learn (paramètre `categorical_features="from_dtype"`) : performances et temps sont-ils comparables ?
- Installez la bibliothèque `shap` et tracez un `shap.summary_plot` (nuage de points des valeurs SHAP colorées par la valeur de la variable).

In [ ]:
# Bonus : choix du seuil


In [ ]:
# Bonus : équité selon le sexe
